In [1]:
import os

os.environ["LIBGL_ALWAYS_SOFTWARE"] = "1"

In [2]:
from pathlib import Path

import nibabel as nib

image_path = Path("/scratch/imaging/ds005237/sub-NDARINVXP963GZ5/func/sub-NDARINVXP963GZ5_task-restPA_run-01_bold.nii.gz")
image = nib.nifti1.load(image_path) # pyright: ignore[reportAttributeAccessIssue]

In [3]:
image.shape

(104, 104, 72, 488)

In [4]:
data = image.get_fdata()
data.shape

(104, 104, 72, 488)

In [5]:
import json

import numpy as np

sidecar = json.loads(
    (image_path.parent / image_path.name.removesuffix(".gz").removesuffix(".nii"))
    .with_suffix(".json")
    .read_text()
)
slice_timing = np.asanyarray(sidecar["SliceTiming"])

In [6]:
slice_timing.max()

np.float64(0.695)

In [7]:
import scipy.stats

slice_ranks = scipy.stats.rankdata(
    (slice_timing * 10000).astype(int), method="dense"
).astype(int)

In [8]:
slice_timing[(slice_ranks == 2)]

array([0.0875, 0.0875, 0.0875, 0.0875, 0.0875, 0.0875, 0.0875, 0.0875])

In [10]:
# delta = np.diff(np.sort(slice_timing, axis=0))
# delta = delta[delta > 0.0]
frames_per_second = np.reciprocal(
    sidecar["RepetitionTime"] / np.unique(slice_ranks).size
)
frames_per_second

np.float64(11.25)

In [11]:
volume_count = round(60 / sidecar["RepetitionTime"])
volume_count

75

In [ ]:
from collections.abc import MutableSequence

import imageio.v3

# import matplotlib.colors
import pyvista as pv
from tqdm.auto import tqdm

# cmap = matplotlib.colors.LinearSegmentedColormap.from_list(
#     "custom", ["#000000", "#00BFFF"]
# )

plot_slices = (
    ("x", "yz"),
    ("y", "xz"),
    ("z", "xy"),
)
plot_voxel = np.array([52, 48, 21])

window_size = [3840, 2160]
aspect = (window_size[0] / len(plot_slices)) / window_size[1]

focal_point = None
volume_center = None
scales: MutableSequence[float | None] = [None, None, None]

writer = imageio.get_writer(
    "raw.webm",
    format="FFMPEG",  # pyright: ignore[reportArgumentType]
    codec="vp9",
    fps=frames_per_second,
    output_params=["-crf", "32"],
)

grid = pv.ImageData()
grid.dimensions = np.array(data.shape[:3]) + 1
grid.spacing = image.header.get_zooms()[:3]
grid.origin = image.affine[:3, 3]

current_data = np.zeros_like(data[..., 0])
cumulative_data = np.zeros_like(data[..., 0])

for volume_index in tqdm(range(volume_count)):  # data.shape[-1]
    for rank in tqdm(np.sort(np.unique(slice_ranks)), leave=False):
        slices = slice_ranks == rank
        current_data *= 0.0
        current_data[..., slices] = 0.25

        cumulative_data[..., slices] = data[..., slices, volume_index]

        grid.cell_data["current"] = current_data.flatten(order="F")
        grid.cell_data["cumulative"] = cumulative_data.flatten(order="F")

        transform_matrix = np.eye(4)
        transformed_grid = grid.transform(transform_matrix, inplace=False)

        if volume_center is None:
            volume_center = (
                transformed_grid.origin + (plot_voxel + 0.5) * transformed_grid.spacing
            )

        plotter = pv.Plotter(
            shape=(1, len(plot_slices)),
            off_screen=True,
            window_size=window_size,
            border=False,
        )
        for i, (normal, camera_position) in enumerate(plot_slices):
            plotter.subplot(0, i)

            plotter.set_background("black")  # pyright: ignore[reportArgumentType]

            intersection = grid.slice(normal=normal, origin=volume_center)  # pyright: ignore[reportArgumentType]
            plotter.add_mesh(
                intersection,
                scalars="cumulative",
                cmap="gray",
                lighting=False,
                show_edges=False,
                show_scalar_bar=False,
            )
            plotter.add_mesh(
                intersection,
                scalars="current",
                opacity="current",  # pyright: ignore[reportArgumentType]
                cmap="Blues",
                clim=(0.0, 0.25),
                lighting=False,
                show_edges=False,
                show_scalar_bar=False,
            )

            edges = intersection.threshold(
                [10.0, np.inf], scalars="cumulative"
            ).extract_all_edges()
            if edges.n_faces:
                plotter.add_mesh(edges, color="white", line_width=0.5)  # pyright: ignore[reportArgumentType]
            plotter.enable_anti_aliasing("ssaa")

            plotter.enable_parallel_projection()  # pyright: ignore[reportCallIssue]

            plotter.camera_position = camera_position  # pyright: ignore[reportAttributeAccessIssue]
            if focal_point is None:
                focal_point = intersection.center
            plotter.camera.focal_point = focal_point

            if scales[i] is None:
                bounds = intersection.bounds
                sizes = [
                    bounds.x_max - bounds.x_min,
                    bounds.y_max - bounds.y_min,
                    bounds.z_max - bounds.z_min,
                ]
                sizes.pop(i)
                scale = max(sizes[1], sizes[0] / aspect) * 0.5 * {"x": 0.9, "y": 0.8, "z": 0.7}[normal]
                scales[i] = scale
            plotter.camera.parallel_scale = scales[i]

        plotter.screenshot("screenshot.png")
        plotter.close()

        frame = imageio.v3.imread("screenshot.png")
        writer.append_data(frame)

writer.close()

  0%|          | 0/75 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

...

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

  0%|          | 0/9 [00:00<?, ?it/s]

In [ ]:
image